# 🧬 Remedia — Hızlı Colab Akışı
Varsayılanlar: 10 molekül, top %10 accurate, `balanced` profil, REINVENT ve benchmark kapalı. GNINA fast/accurate aşamalarını batch çalıştırır; SDF'ler bir kez hazırlanır. Pocket merkezi Drive'da UniProt ID ile cache'lenir.

In [ ]:
#@title 1 — Hedef, cache ve kurulum { display-mode: "form" }
REPO_REF = 'main'  #@param {type:"string"}
UNIPROT_ID = 'P00918'  #@param {type:"string"}
BOX_DIM = 20  #@param {type:"slider", min:10, max:30, step:1}
USE_CACHED_POCKET = True  #@param {type:"boolean"}
FORCE_REFRESH_POCKET = False  #@param {type:"boolean"}

import json, subprocess, sys
from pathlib import Path
BOX_SIZE=(float(BOX_DIM),)*3
DRIVE_SETUP_DIR=None
try:
 from google.colab import drive
 drive.mount('/content/drive',force_remount=False)
 DRIVE_SETUP_DIR=Path('/content/drive/MyDrive/remedia_setup'); DRIVE_SETUP_DIR.mkdir(parents=True,exist_ok=True)
except Exception as e: print('Drive yok:',e)
POCKET_CACHE_PATH=DRIVE_SETUP_DIR/'pocket_cache.json' if DRIVE_SETUP_DIR else None
pocket_cache={}
if POCKET_CACHE_PATH and POCKET_CACHE_PATH.exists():
 try: pocket_cache=json.loads(POCKET_CACHE_PATH.read_text())
 except Exception: pass
CACHE_KEY=UNIPROT_ID.strip().upper()
CACHED_POCKET=pocket_cache.get(CACHE_KEY) if USE_CACHED_POCKET and not FORCE_REFRESH_POCKET else None
NEED_FPOCKET=CACHED_POCKET is None
if not NEED_FPOCKET:
 print('⚡ Pocket cache bulundu; Miniconda/fpocket atlanacak:',CACHED_POCKET['center'])
else:
 try:
  import condacolab; condacolab.check(); print('✅ Miniconda hazır')
 except Exception:
  subprocess.run([sys.executable,'-m','pip','install','-q','condacolab'],check=True)
  import condacolab
  print('Kernel yeniden başlayınca Run all çalıştır.'); condacolab.install()


In [ ]:
#@title 2 — Ortam, hedef ve tohumlar { display-mode: "form" }
import datetime, os, shutil, stat, subprocess, sys, time, urllib.request
from pathlib import Path
REPO_DIR=Path('Remedia')
if not REPO_DIR.is_dir(): subprocess.run(['git','clone','-q','--branch',REPO_REF,'https://github.com/mehmetg06/Remedia.git'],check=True)
else:
 subprocess.run(['git','-C',str(REPO_DIR),'fetch','-q','origin',REPO_REF],check=False)
 subprocess.run(['git','-C',str(REPO_DIR),'checkout','-q',REPO_REF],check=False)
 subprocess.run(['git','-C',str(REPO_DIR),'pull','-q','--ff-only'],check=False)
SRC_DIR=(REPO_DIR/'src').resolve(); DATA_DIR=(REPO_DIR/'data').resolve(); sys.path.insert(0,str(SRC_DIR))
for mod,pkg in [('rdkit','rdkit'),('requests','requests'),('pandas','pandas'),('tqdm','tqdm'),('meeko','meeko')]:
 try: __import__(mod)
 except ImportError: subprocess.run([sys.executable,'-m','pip','install','-q',pkg],check=True)
if NEED_FPOCKET and shutil.which('fpocket') is None:
 subprocess.run('conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main || true',shell=True)
 subprocess.run('conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r || true',shell=True)
 subprocess.run('conda install -y -c conda-forge -c bioconda fpocket >/dev/null 2>&1',shell=True,check=False)
GNINA_PATH='/usr/local/bin/gnina'; cache=DRIVE_SETUP_DIR/'gnina' if DRIVE_SETUP_DIR else None
if not (Path(GNINA_PATH).exists() and Path(GNINA_PATH).stat().st_size>1_000_000):
 if cache and cache.exists(): shutil.copy(cache,GNINA_PATH)
 else:
  urllib.request.urlretrieve('https://github.com/gnina/gnina/releases/download/v1.3/gnina',GNINA_PATH)
  if cache: shutil.copy(GNINA_PATH,cache)
os.chmod(GNINA_PATH,0o755)
if subprocess.run(['nvidia-smi'],capture_output=True).returncode: raise RuntimeError('T4 GPU seç')
RUN_ID=datetime.datetime.now().strftime('run_%Y%m%d_%H%M%S'); RESULTS_DIR=Path('results')/RUN_ID; RESULTS_DIR.mkdir(parents=True)
import fetch_structure, pocket_detection
pdb_path=fetch_structure.fetch_alphafold(CACHE_KEY); RECEPTOR=str(pdb_path); CENTER=None
if CACHED_POCKET: CENTER=tuple(CACHED_POCKET['center'])
elif shutil.which('fpocket'):
 try:
  p=pocket_detection.best_druggable_pocket(pdb_path); CENTER=tuple(round(x,3) for x in p['center'])
  pocket_cache[CACHE_KEY]={'center':list(CENTER),'source':'fpocket','druggability':p.get('druggability')}
  if POCKET_CACHE_PATH: POCKET_CACHE_PATH.write_text(json.dumps(pocket_cache,indent=2))
 except Exception as e: print('fpocket başarısız:',e)
if CENTER is None:
 xyz=[(float(l[30:38]),float(l[38:46]),float(l[46:54])) for l in Path(pdb_path).read_text().splitlines() if l.startswith(('ATOM','HETATM'))]
 CENTER=tuple(round(sum(a)/len(xyz),3) for a in zip(*xyz)); print('⚠️ Geometrik fallback; cache edilmedi')
from known_ligands import fetch_known_ligands
from rdkit import Chem
ligands,msg=fetch_known_ligands(CACHE_KEY,max_results=5); print(msg)
seeds=[x['smiles'] for x in ligands] or ['NS(=O)(=O)c1ccccc1','CC(=O)Nc1nnc(s1)S(N)(=O)=O']
seeds=[s for s in seeds if Chem.MolFromSmiles(s)]
print('✅ center=',CENTER,'seeds=',len(seeds),'results=',RESULTS_DIR)


In [ ]:
#@title 3 — Üretim ve batch GNINA { display-mode: "form" }
method = 'fusion'  #@param ["fusion", "genetic", "brics", "random", "pretrained"]
GENERATE_N = 10  #@param {type:"slider", min:5, max:100, step:5}
GA_GENERATIONS = 3  #@param {type:"slider", min:2, max:20, step:1}
INSTALL_REINVENT4 = False  #@param {type:"boolean"}
DOCKING_MODE = 'iki_asamali'  #@param ["iki_asamali", "sadece_fast", "sadece_accurate"]
ACCURACY_PROFILE = 'balanced'  #@param ["balanced", "final"]
TOP_FRACTION = 0.10  #@param {type:"slider", min:0.05, max:0.50, step:0.05}
RUN_BENCHMARK = False  #@param {type:"boolean"}
BENCHMARK_N = 3  #@param {type:"slider", min:2, max:10, step:1}

from molecule_generator import fusion_generation,genetic_algorithm,random_mutation,brics_recombination,write_smi
if INSTALL_REINVENT4:
 from generative_model import install_reinvent
 install_reinvent(log_fn=print,drive_cache_dir=DRIVE_SETUP_DIR)
if method=='fusion': final,_=fusion_generation(seeds,docking_opts=None,log_fn=print,population_size=max(10,GENERATE_N),generations=GA_GENERATIONS); generated=[s for s,_ in final]
elif method=='genetic': final,_=genetic_algorithm(seeds,generations=GA_GENERATIONS,population_size=max(10,GENERATE_N),docking_opts=None,log_fn=print); generated=[s for s,_ in final]
elif method=='brics': generated=brics_recombination(seeds,n=GENERATE_N)
elif method=='random': generated=random_mutation(seeds,n=GENERATE_N)
else:
 from generative_model import generate_with_reinvent
 generated=generate_with_reinvent(num_molecules=GENERATE_N,output_path=RESULTS_DIR/'generated.smi',drive_cache_dir=DRIVE_SETUP_DIR)
if method!='pretrained' and len(generated)<GENERATE_N:
 pool=set(generated); pool.update(random_mutation(seeds,n=GENERATE_N)); pool.update(brics_recombination(seeds,n=GENERATE_N)); generated=list(pool)
generated=[s for s in generated if s][:GENERATE_N]; molecules=[(f'mol_{i:03d}',s) for i,s in enumerate(generated,1)]
smi_out=RESULTS_DIR/'generated.smi'
if method!='pretrained': write_smi(generated,smi_out,prefix='mol')
import gnina_engine, pandas as pd
common=dict(receptor=RECEPTOR,center=CENTER,size=BOX_SIZE,gnina_path=GNINA_PATH,out_dir=RESULTS_DIR/'gnina_out',log_fn=print,profile=ACCURACY_PROFILE)
if DOCKING_MODE=='iki_asamali': rows,stage_info=gnina_engine.run_two_stage_screening(molecules,top_fraction=TOP_FRACTION,**common)
elif DOCKING_MODE=='sadece_fast': rows,stage_info=gnina_engine.run_single_mode_screening(molecules,mode='fast',**common)
else: rows,stage_info=gnina_engine.run_single_mode_screening(molecules,mode='accurate',**common)
docking_df=pd.DataFrame(rows); display(docking_df)
print('✅ GNINA süreç sayısı:',stage_info.get('gnina_processes'))
if RUN_BENCHMARK:
 bench,summary=gnina_engine.benchmark_fast_vs_accurate(molecules[:BENCHMARK_N],receptor=RECEPTOR,center=CENTER,size=BOX_SIZE,gnina_path=GNINA_PATH,out_dir=RESULTS_DIR/'benchmark',profile=ACCURACY_PROFILE)
 pd.DataFrame(bench).to_csv(RESULTS_DIR/'benchmark.csv',index=False); print(summary)
else: print('⏭️ Benchmark kapalı')


In [ ]:
#@title 4 — Filtre, sıralama ve kayıt { display-mode: "form" }
import shutil, subprocess, sys, pandas as pd
import admet_filter
admet_df=pd.DataFrame([admet_filter.lipinski_veber_filter(s,n) for n,s in molecules])
docking_csv=RESULTS_DIR/'docking_scores.csv'; admet_csv=RESULTS_DIR/'admet_results.csv'; ranked_csv=RESULTS_DIR/'final_ranking.csv'
docking_df.to_csv(docking_csv,index=False); admet_df.to_csv(admet_csv,index=False)
subprocess.run([sys.executable,str(SRC_DIR/'rank_report.py'),'--docking',str(docking_csv),'--admet',str(admet_csv),'--output',str(ranked_csv)],check=True)
ranked_df=pd.read_csv(ranked_csv); display(ranked_df.head(10))
if DRIVE_SETUP_DIR:
 save_dir=Path('/content/drive/MyDrive/Remedia_results')/RUN_ID; save_dir.mkdir(parents=True,exist_ok=True)
 for f in [smi_out,docking_csv,admet_csv,ranked_csv]:
  if Path(f).exists(): shutil.copy(f,save_dir/Path(f).name)
 print('✅ Drive:',save_dir)
